# CS 3110/5110: Data Privacy

## Homework 8


In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
import random
from scipy import stats
import matplotlib.pyplot as plt


def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)


def laplace_mech_vec(vec, sensitivity, epsilon):
    return [
        v + np.random.laplace(loc=0, scale=sensitivity / epsilon) for v in vec
    ]


def gaussian_mech(v, sensitivity, epsilon, delta):
    return v + np.random.normal(
        loc=0, scale=sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon
    )


def gaussian_mech_vec(vec, sensitivity, epsilon, delta):
    return [
        v
        + np.random.normal(
            loc=0,
            scale=sensitivity * np.sqrt(2 * np.log(1.25 / delta)) / epsilon,
        )
        for v in vec
    ]


def pct_error(orig, priv):
    return np.abs(orig - priv) / orig * 100.0


adult = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv"
)

In [2]:
# preserves epsilon-differential privacy
def above_threshold(queries, df, T, epsilon):
    T_hat = T + np.random.laplace(loc=0, scale=2 / epsilon)

    for idx, q in enumerate(queries):
        nu_i = np.random.laplace(loc=0, scale=4 / epsilon)
        if q(df) + nu_i >= T_hat:
            return idx
    return -1  # the index of the last element

## Question 1 (20 points)

Implement a function `above_10000` that releases the **value** of the first query in a sequence of queries whose value is above 10000. Your function should have a **total** privacy cost equal to the privacy parameter $\epsilon$ passed in when it is called.


In [3]:
def make_query(status):
    def q(df):
        return len(df[df["Marital Status"] == status])

    return q


def above_10000(queries, epsilon):
    # Above threshold will return the INDEX of the first query to exceed the
    # hardcoded 10,000 value.
    query_index = above_threshold(queries, adult, T=10000, epsilon=epsilon / 2)
    # The query index has noise added to it via the above_threshold mechanism.

    # NB: the value of the query itself needs noise added to it as well! So we
    # use the laplace mechanism to add noise to the query result.
    # The query is a counting query so sensitivity is 1.
    return laplace_mech(
        queries[query_index](adult), sensitivity=1, epsilon=epsilon / 2
    )


queries = [make_query(status) for status in adult["Marital Status"].unique()]
print(f"above_10000 #1: {above_10000(queries, 100)}")
print(f"above_10000 #2: {above_10000(queries, 1)}")
print(f"above_10000 #3: {above_10000(queries, .01)}")

above_10000 #1: 10683.029710034189
above_10000 #2: 10682.585067885548
above_10000 #3: 10949.32665109164


In [4]:
# TEST CASE

results = [above_10000(queries, 1.0) for _ in range(100)]
print(np.mean(results))
assert np.mean(results) > 9900
assert np.mean(results) < 11000

10682.761013639052


## Question 2 (10 points)

In 2-3 sentences, argue informally (via the definition of the sparse vector technique, post-processing, and sequential composition), that your implementation of `above_10000` has a total privacy cost of $\epsilon$.


The list of potential queries are all counting queries, which means their
individual sensitivities are all 1. Because they all possess sensitivities of 1,
we may use the Sparse Vector Technique to _noisily_ select the first query from
the list that satisfies the 10,000 threshold value. This is implemented using
an epsilon value of $\epsilon/2$.

Next, we conduct the chosen query on the dataset, and apply the laplace
mechanism to the result using an epsilon value of $\epsilon/2$.

These two mechanisms are applied in order, thus sequential composition means the
total privacy cost is $\epsilon/2 + \epsilon/2 = \epsilon$. The returned value
has had noise applied to it so post-processing means no addition noise is necessary.


## Question 3 (20 points)

Implement a function `bounded_all_above_10000` that releases the **value** of **$c$ queries** in a sequence of queries whose value is above 10000 (where $c$ is an analyst-provided parameter limiting the number of returned results). Your function should have a **total privacy cost** bounded by its parameter $\epsilon$.


In [5]:
def bounded_all_above_10000(queries, c, epsilon):
    # Because the query values are in order, once we know the index of the first
    # one above 10000, we know the rest are too, and can return the first "c" of
    # them with noise.

    index = above_threshold(queries, adult, T=10000, epsilon=epsilon / (c + 1))

    # Now that we have the starting index, return the first c with noise.
    return [
        laplace_mech(
            queries[i](adult), sensitivity=1, epsilon=epsilon / (c + 1)
        )
        for i in range(index, min(index + c, len(queries)))
    ]
    # adding in min() to avoid c going past the list length


queries = [make_query(status) for status in adult["Marital Status"].unique()]
print(f"bounded_all_above_10000 #1: {bounded_all_above_10000(queries, 3, 100)}")
print(f"bounded_all_above_10000 #2: {bounded_all_above_10000(queries, 3, 1)}")
print(f"bounded_all_above_10000 #3: {bounded_all_above_10000(queries, 3, .01)}")

bounded_all_above_10000 #1: [10683.030338418377, 14975.947027586963, 4442.938593548946]
bounded_all_above_10000 #2: [10681.338097768463, 14972.699928714197, 4443.6400343850955]
bounded_all_above_10000 #3: [10413.240942126711, 15072.215680727095, 4922.753452590527]


In [6]:
# TEST CASE

results = [bounded_all_above_10000(queries, 2, 1.0) for _ in range(100)]
results_1 = [r[0] for r in results]
results_2 = [r[1] for r in results]

assert np.mean(results_1) > 9900
assert np.mean(results_1) < 11000
assert np.mean(results_2) > 14000
assert np.mean(results_2) < 15500

## Question 4 (10 points)

In 2-3 sentences, argue informally that your implementation of `bounded_all_above_10000` has privacy cost bounded by $\epsilon$.


Similarly to the prior question, the query index is noisily chosen via the Sparse Vector
Technique, but this time the epsilon value used is $\epsilon/(c+1)$.

After computing the index, we perform $c$ sequential noisy queries, each with an
epsilon value of $\epsilon/(c+1)$.

By post-processing all of the returned query results have had noise added to
them. Every noise application mechanism uses an epsilon value of
$\epsilon/(c+1)$, and there were exactly $(c+1)$ individual applications (The
Sparse Vector Technique, and then $c$ laplace mechanisms) so the total privacy
cost is $(c+1) * \epsilon/(c+1) = \epsilon$.


## Question 5 (30 points)

Implement a function `mean_age` that computes the mean age of participants in the `adult_data` dataset. Your function should have a **total** privacy cost of $\epsilon$. It should work as follows:

1. Compute an upper clipping parameter based on the data using a differentially private algorithm
2. Clip the data using the clipping parameter
3. Use `laplace_mech` to release a differentially private mean of the clipped data

_Hint_: Use the sparse vector technique (`above_threshold`) to compute the clipping parameter. Consider using a sequence of queries that looks like `df.clip(lower=0, upper=b).sum() - df.clip(lower=0, upper=b+1).sum()`.

_Hint_: Be careful of sensitivities and set the scale of the noise accordingly!


In [7]:
bs = list(range(0, 200, 10))
df = adult["Age"]
b_lower = 0


def make_query(b):
    def q(df):
        # return a positive value if I have reached the "plateau" of the b
        # graph.
        sum1 = df.clip(lower=b_lower, upper=b).sum()
        sum2 = df.clip(lower=b_lower, upper=b + 1).sum()
        return sum1 - sum2
        # The sensitivity here is 1, since changing one row changes sum by at
        # most 1. Not easily seen (since it's not a simple counting query) but
        # still satisfies the Above Threshold condition of the queries needing
        # sensitivity of 1.

    return q


def pick_b(epsilon):
    queries = [make_query(b) for b in bs]
    b_idx = above_threshold(queries, df, T=0, epsilon=epsilon)
    return bs[b_idx]


def mean_age(epsilon):
    # Use our pick_b to pick our bounding parameter b, using only a fraction of
    # the privacy budget.
    b_upper = pick_b(epsilon * 2 / 4)

    # use our clipping parameter to clip the ages and get a noisy sum.
    clipped_ages = df.clip(lower=b_lower, upper=b_upper)
    sum_noisy = laplace_mech(
        clipped_ages.sum(),
        sensitivity=b_upper,
        epsilon=(epsilon * 1 / 4),
    )

    # get a noisy count
    count_noisy = laplace_mech(
        len(clipped_ages),
        sensitivity=1,
        epsilon=(epsilon * 1 / 4),
    )

    # return the noisy mean
    return sum_noisy / count_noisy


for epsilon in [0.001, 0.01, 0.1, 0.5, 1, 10]:
    print(f"epsilon: {epsilon}, mean age: {mean_age(epsilon)}")

epsilon: 0.001, mean age: 19.445727500336627
epsilon: 0.01, mean age: 37.47502504406905
epsilon: 0.1, mean age: 38.59450944015214
epsilon: 0.5, mean age: 38.75933425662769
epsilon: 1, mean age: 38.58776003206841
epsilon: 10, mean age: 38.58155859344329


In [8]:
# TEST CASE
results = [mean_age(1.0) for _ in range(100)]
assert np.mean(results) > 38
assert np.mean(results) < 39

## Question 6 (10 points)

In 3-5 sentences, describe your approach to implementing `mean_age` and argue informally that your implementation has privacy cost bounded by $\epsilon$.


We start by creating a sequence of queries that test various values of `b`, with
each query testing if going from `b` to `b + 1` would cause the sum query to not
increase (therefore decrease or stay at zero). Using the Sparse Vector Technique
we are able to find a chosen `b` using an epsilon cost of $\epsilon*2/4$.

Next, we clip all the ages and use the laplace mechanism to get a noisy sum using
the newly chosen clipping parameter `b` and an epsilon value of $\epsilon/4$.

Then we get a noisy count using an epsilon value of $\epsilon/4$ and a
sensitivity value of 1.

Lastly, we are able to compute and return a noisy average using the noisy sum
and noisy count for free via post-processing.

The three mechanisms used are combined via sequential composition to produce a
total privacy cost of $\epsilon*2/4 + \epsilon/4 + \epsilon/4 = \epsilon$.


## Question 7 (20 points)

Write an algorithm to compute the (approximate) maximum of a given column of the adult dataset. Your solution should satisfy `epsilon` differential privacy.

_Hint_: you might want to use AboveThreshold; you might need to test several options for building your queries and for the threshold


In [9]:
candidate_values_max_steps = 1000
geometric_factor = 1.1
# Possible max values to test. These increase geometrically to cover a wide
# range of test values.
candidate_values = [
    geometric_factor**k for k in range(candidate_values_max_steps)
]


def make_query(column, candidate):
    def q(df):
        # return a positive value if I have reached the "plateau" of the sums.
        sum1 = df[column].clip(lower=0, upper=candidate).sum()
        sum2 = df[column].clip(lower=0, upper=candidate + 1).sum()
        return sum1 - sum2

    return q


def max_of_col(column, epsilon):
    queries = [make_query(column, candidate) for candidate in candidate_values]
    candidate_idx = above_threshold(queries, adult, T=0, epsilon=epsilon)
    return candidate_values[candidate_idx]


max_of_col("Age", 1.0)

97.01723378487254

In [10]:
# TEST CASE
cols = ["Age", "Capital Gain", "fnlwgt"]

for c in cols:
    true_val = adult[c].max()
    trials = [max_of_col(c, 10.0) for _ in range(20)]
    errors = [pct_error(true_val, t) for t in trials]
    print("Median error for column " + c + ":", np.median(errors))
    assert np.median(errors) > 0
    assert np.median(errors) < 100

Median error for column Age: 18.576619070399783
Median error for column Capital Gain: 1.9809955095691598
Median error for column fnlwgt: 14.406816678398744


## Question 8 (10 points)

In 3-5 sentences, describe your approach to implementing `max_of_col` and argue that your approach has total privacy cost bounded by $\epsilon$.


Since we are generalizing to any column here, and can no longer guarantee that the max value is
within any given range, we need a more generic approach. Therefore, I increase
our test values in between the test queries geometrically instead of
arithmetically, so that we can more reasonably test a larger range of values.

The test queries use the test values as clipping parameters and calculate if the
sum value did not increase when the test clipping parameter increased (which
would imply we have surpassed the maximum value in the given column). This is an
application of the Sparse Vector Technique, and we do this with an epsilon value
of $\epsilon$. Returning the found max value incurs no additional cost via
post-processing, and the value is computed via above_threshold with an epsilon
value of $\epsilon$, therefore we are successfully differentially private and
our total cost is only $\epsilon$.
